# Aula 2 — Revisão de Probabilidade

**Laboratórios de Econometria em R** · MPEF/FGV-EPGE · Prof. Marcelo Mello

---

A regressão é um método para estimar **médias condicionais**. Antes de construí-la,
vale fixar as ferramentas probabilísticas de que ela depende.

Quatro resultados:

1. a Binomial dos pênaltis, e o que é uma distribuição de probabilidade;
2. a desigualdade de **Chebyshev** como cota válida para qualquer distribuição;
3. a **lei das expectativas iteradas**, verificada numericamente;
4. o contraexemplo decisivo: **covariância zero sem independência**.

In [ ]:
for (p in c("ggplot2", "dplyr")) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)
}
library(ggplot2)
suppressMessages(library(dplyr))
theme_set(theme_minimal(base_size = 13))
az <- "#1f4e79"; vd <- "#2e7d32"; vm <- "#b3261e"; cz <- "grey45"
options(repr.plot.width = 8, repr.plot.height = 4)

## 1. A Binomial: a disputa de pênaltis

Cinco cobranças, probabilidade 0,75 de conversão em cada, independentes. Quantos gols?
A distribuição de $X \sim \text{Binomial}(5;\,0{,}75)$ dá a resposta exata.

In [ ]:
n_pen <- 5; p_gol <- 0.75
k <- 0:n_pen

dist <- data.frame(
  gols        = k,
  probabilidade = dbinom(k, n_pen, p_gol),
  acumulada     = pbinom(k, n_pen, p_gol)
)
round(dist, 4)

In [ ]:
# média e variância: fórmula fechada contra a definição
c(media_formula   = n_pen * p_gol,
  media_definicao = sum(k * dbinom(k, n_pen, p_gol)),
  var_formula     = n_pen * p_gol * (1 - p_gol),
  var_definicao   = sum((k - n_pen * p_gol)^2 * dbinom(k, n_pen, p_gol)))

In [ ]:
ggplot(dist, aes(x = factor(gols), y = probabilidade)) +
  geom_col(fill = az, width = 0.65) +
  geom_text(aes(label = sprintf("%.3f", probabilidade)), vjust = -0.4, size = 3.6) +
  labs(x = "gols em 5 cobranças", y = "probabilidade") +
  expand_limits(y = max(dist$probabilidade) * 1.12)

## 2. Chebyshev: uma cota que vale sempre

A desigualdade de Chebyshev afirma que, para **qualquer** distribuição com variância
finita,

$$P\left(|X - \mu| \geq c\,\sigma\right) \leq \frac{1}{c^2}.$$

O preço da generalidade é que a cota costuma ser **folgada**. Vamos medir a folga em
cinco distribuições bem diferentes.

In [ ]:
set.seed(451)
N <- 200000
amostras <- list(
  Normal      = rnorm(N),
  Uniforme    = runif(N, -1, 1),
  Exponencial = rexp(N, 1),
  `t(5)`      = rt(N, df = 5),
  Bernoulli   = rbinom(N, 1, 0.5)
)

cs <- c(1.5, 2, 3)

# uma LINHA por valor de c: as frequências observadas em cada distribuição,
# seguidas da cota de Chebyshev — que depende só de c, não da distribuição.
freq <- sapply(amostras, function(x) {
  z <- abs(x - mean(x)) / sd(x)
  sapply(cs, function(c) mean(z >= c))
})
tabela <- cbind(freq, `cota 1/c^2` = 1 / cs^2)
rownames(tabela) <- paste0("c = ", cs)
round(tabela, 4)

Em cada linha, todas as distribuições ficam **abaixo** da última coluna — como o
teorema garante. Mas a folga é enorme: para $c=2$, Chebyshev permite até 25% e a
Normal entrega 4,6%. Chebyshev é uma **garantia mínima válida para qualquer
distribuição**, não uma aproximação — e é justamente por não supor nada que ela
custa tão caro.

## 3. A lei das expectativas iteradas

$$\mathbb{E}(Y) = \mathbb{E}\left[\mathbb{E}(Y \mid X)\right]$$

A média não condicional é a **média ponderada** das médias condicionais, com pesos
iguais às probabilidades de cada valor de $X$. Verificamos na tabela chuva × tempo de
deslocamento.

In [ ]:
# distribuição conjunta: X = chove (0/1), Y = tempo de deslocamento em minutos
conj <- expand.grid(chuva = c(0, 1), tempo = c(30, 45, 60))
conj$prob <- c(0.30, 0.06,    # tempo 30
               0.24, 0.14,    # tempo 45
               0.06, 0.20)    # tempo 60
stopifnot(abs(sum(conj$prob) - 1) < 1e-12)

# marginais
p_chuva <- tapply(conj$prob, conj$chuva, sum)
p_tempo <- tapply(conj$prob, conj$tempo, sum)
list(marginal_chuva = round(p_chuva, 3), marginal_tempo = round(p_tempo, 3))

In [ ]:
# médias condicionais E(Y | X = x)
med_cond <- sapply(c(0, 1), function(x) {
  sub <- conj[conj$chuva == x, ]
  sum(sub$tempo * sub$prob) / sum(sub$prob)
})
names(med_cond) <- c("sem chuva", "com chuva")

esperanca_direta   <- sum(conj$tempo * conj$prob)
esperanca_iterada  <- sum(med_cond * p_chuva)

round(c(med_cond,
        E_direta  = esperanca_direta,
        E_iterada = esperanca_iterada,
        diferenca = esperanca_direta - esperanca_iterada), 6)

As duas formas de calcular $\mathbb{E}(Y)$ coincidem até a precisão da máquina. Note
também que $\mathbb{E}(Y \mid \text{chuva}) > \mathbb{E}(Y \mid \text{sem chuva})$:
saber que choveu **muda** o palpite sobre o tempo de viagem — é exatamente o que a
regressão vai explorar.

## 4. Covariância zero **não** é independência

Este é o contraexemplo que organiza toda a leitura de correlações no curso.

In [ ]:
set.seed(77)
x <- runif(30000, -1, 1)
y <- x^2 + rnorm(30000, sd = 0.05)   # y é FUNÇÃO de x — dependência perfeita, quase

c(correlacao = cor(x, y),
  covariancia = cov(x, y))

In [ ]:
# ...e no entanto a média condicional varia fortemente com x
faixas <- cut(x, breaks = seq(-1, 1, by = 0.25))
med <- tapply(y, faixas, mean)
round(med, 3)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 3.8)
d <- data.frame(x, y)
ggplot(d[sample(nrow(d), 4000), ], aes(x, y)) +
  geom_point(alpha = 0.16, colour = cz, size = 0.8) +
  geom_smooth(method = "lm", se = FALSE, colour = vm, linewidth = 1) +
  stat_summary_bin(fun = mean, bins = 16, geom = "line",
                   colour = az, linewidth = 1.1) +
  labs(x = "x", y = "y",
       subtitle = "vermelho: reta de MQO (inclinação ~0) · azul: média condicional real")

A correlação é praticamente zero e a **reta de MQO é plana** — mas $y$ é uma função
determinística de $x$. A lição:

> **Correlação mede associação _linear_.** Ausência de correlação não é ausência de
> relação. A independência é uma condição bem mais forte.

É por isso que a hipótese de identificação do curso é
$\mathbb{E}(u \mid X) = 0$ — sobre a **média condicional** — e não meramente
$\text{Cov}(X, u) = 0$.

## Para experimentar

1. Na Binomial, faça `p_gol <- 0.5` e veja a distribuição ficar simétrica.
2. Em Chebyshev, acrescente uma distribuição de cauda pesada (`rt(N, df = 2)`) e
   compare a folga.
3. No contraexemplo, troque `x^2` por `sin(3*x)` — a correlação continua perto de
   zero?

---

⬅️ [Aula 1](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/01-causalidade.ipynb) · [🏠 Índice](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/00-indice.ipynb) · ➡️ [**Aula 3 — Regressão simples**](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/03-regressao-simples.ipynb)